# EDA — Airbnb & Kamernet raw data

Exploratory look at the raw sources before any Bronze/Silver cleaning.
Goal: understand distributions, missingness, and outliers so the
Silver-layer cleaning decisions (price capping, zipcode backfill,
numeric parsing) are grounded in evidence.

Uses pandas + matplotlib — Spark is overkill for a ~50k-row EDA, and
the goal here is interactive exploration, not productionisation.

In [ ]:
import json
import os
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)
plt.rcParams["figure.dpi"] = 110

DATA_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", "data"))
AIRBNB_CSV = f"{DATA_ROOT}/airbnb.csv"
RENTALS_JSON = f"{DATA_ROOT}/rentals.json"
print("Data root:", DATA_ROOT)

## 1. Airbnb

Expected to be short-term listing snapshots (one row per listing) with
nightly price in EUR.  Numeric fields of interest: `price`, `accommodates`,
`bedrooms`, `review_scores_value`, plus lat/long for geo sanity.

In [ ]:
if os.path.exists(AIRBNB_CSV):
    airbnb = pd.read_csv(AIRBNB_CSV)
else:
    with zipfile.ZipFile(AIRBNB_CSV.replace(".csv", ".zip")) as zf:
        airbnb = pd.read_csv(zf.open("airbnb.csv"))

print(airbnb.shape)
airbnb.head()

In [ ]:
# Schema & missingness
info = pd.DataFrame(
    {
        "dtype": airbnb.dtypes.astype(str),
        "n_missing": airbnb.isna().sum(),
        "pct_missing": (airbnb.isna().mean() * 100).round(2),
        "n_unique": airbnb.nunique(),
    }
)
info

In [ ]:
# Descriptive statistics for the numeric columns
num_cols_airbnb = [
    "price",
    "accommodates",
    "bedrooms",
    "review_scores_value",
    "latitude",
    "longitude",
]
airbnb[num_cols_airbnb].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).round(2)

### 1a. Outlier detection — IQR & z-score

- **IQR rule:** flag anything outside `[Q1 - 1.5·IQR, Q3 + 1.5·IQR]`.
  Robust to the heavy right-tail typical of listing prices.
- **z-score rule:** flag `|z| > 3`.  Sensitive to extreme values but useful
  as a sanity check — if IQR flags 5% but z-score flags 0.1%, the tail is
  long-but-dense rather than a few freak values.

In [ ]:
def outlier_report(s: pd.Series) -> dict:
    s = s.dropna()
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    iqr_mask = (s < lo) | (s > hi)
    z = (s - s.mean()) / s.std(ddof=0)
    z_mask = z.abs() > 3
    return {
        "n": len(s),
        "q1": round(q1, 2),
        "q3": round(q3, 2),
        "iqr_lower_fence": round(lo, 2),
        "iqr_upper_fence": round(hi, 2),
        "n_iqr_outliers": int(iqr_mask.sum()),
        "pct_iqr_outliers": round(iqr_mask.mean() * 100, 2),
        "n_zscore_outliers": int(z_mask.sum()),
        "pct_zscore_outliers": round(z_mask.mean() * 100, 2),
        "max": round(s.max(), 2),
        "min": round(s.min(), 2),
    }


airbnb_outliers = pd.DataFrame({c: outlier_report(airbnb[c]) for c in num_cols_airbnb}).T
airbnb_outliers

### 1b. IQR vs z-score — agreement crosstab

The two methods flag *different* rows, not just different counts. A 2x2
crosstab shows the overlap on Airbnb `price`:

- Diagonal = both methods agree (clear inliers / clear outliers).
- **Off-diagonal `(IQR=True, z=False)`** = rows IQR catches that z-score
  misses. If this cell is large, the distribution is right-skewed and
  z-score is under-flagging because the tail inflates `std`. That's the
  evidence for picking IQR as the cleaning rule in Silver.

In [ ]:
s = airbnb["price"].dropna()
q1, q3 = s.quantile([0.25, 0.75])
iqr = q3 - q1
iqr_flag = (s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)
z_flag = ((s - s.mean()) / s.std(ddof=0)).abs() > 3

pd.crosstab(iqr_flag, z_flag, rownames=["IQR outlier"], colnames=["z outlier"])

In [ ]:
# Price — histogram (raw + log) and boxplot.
# The raw histogram is dominated by the tail; the log view makes the
# right-skew visible, confirming why we cap rather than drop outliers.
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(airbnb["price"].dropna(), bins=80, color="steelblue")
axes[0].set_title("Airbnb price (EUR / night) — raw")
axes[0].set_xlabel("price")

axes[1].hist(np.log1p(airbnb["price"].dropna()), bins=60, color="steelblue")
axes[1].set_title("Airbnb price — log1p")
axes[1].set_xlabel("log1p(price)")

axes[2].boxplot(airbnb["price"].dropna(), vert=True)
axes[2].set_title("Airbnb price — boxplot")
axes[2].set_ylabel("price")

plt.tight_layout()
plt.show()

In [ ]:
# Boxplots for the other numeric fields — quick scan for suspicious tails
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, col in zip(axes, ["accommodates", "bedrooms", "review_scores_value", "latitude"]):
    ax.boxplot(airbnb[col].dropna(), vert=True)
    ax.set_title(col)
plt.tight_layout()
plt.show()

In [ ]:
# Top 10 price outliers — are they plausible (luxury villas) or data errors?
airbnb.nlargest(10, "price")[["zipcode", "room_type", "accommodates", "bedrooms", "price"]]

In [ ]:
# Geo sanity: Amsterdam is roughly lat 52.3-52.4, lon 4.75-5.0
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(airbnb["longitude"], airbnb["latitude"], s=2, alpha=0.3, color="steelblue")
ax.set_title("Airbnb listings — spatial distribution")
ax.set_xlabel("longitude")
ax.set_ylabel("latitude")
plt.show()

off_bbox = airbnb[
    (airbnb["latitude"] < 52.3)
    | (airbnb["latitude"] > 52.45)
    | (airbnb["longitude"] < 4.7)
    | (airbnb["longitude"] > 5.05)
]
print(f"Listings outside Amsterdam bbox: {len(off_bbox)}")

## 2. Kamernet rentals

Long-term listings.  Several fields arrive as single-element lists
(e.g. `_id`, `crawledAt`) and currency/area fields are formatted strings
(`"€ 1.500,-"`, `"45 m²"`) — so we parse them here just enough to run
descriptive stats.

In [ ]:
if os.path.exists(RENTALS_JSON):
    with open(RENTALS_JSON) as f:
        rentals_raw = json.load(f)
else:
    with zipfile.ZipFile(RENTALS_JSON.replace(".json", ".zip")) as zf:
        rentals_raw = json.load(zf.open("rentals.json"))

rentals = pd.DataFrame(rentals_raw)
print(rentals.shape)
rentals.head(2)

In [ ]:
# City distribution — assessment scope is Amsterdam only
rentals["city"].value_counts().head(10)

In [ ]:
# Scope the EDA to Amsterdam so distributions are comparable to the pipeline output
rentals_ams = rentals[rentals["city"] == "Amsterdam"].copy()
print(f"Amsterdam records: {len(rentals_ams):,} of {len(rentals):,}")

In [ ]:
# Parse currency and area strings
import re


def parse_eur(x):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return np.nan
    s = str(x)
    m = re.search(r"([0-9][0-9\.\,]*)", s)
    if not m:
        return np.nan
    # NL formatting: thousands = '.', decimal = ','
    num = m.group(1).replace(".", "").replace(",", ".")
    try:
        return float(num)
    except ValueError:
        return np.nan


def parse_sqm(x):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return np.nan
    m = re.search(r"(\d+)", str(x))
    return float(m.group(1)) if m else np.nan


rentals_ams["rent_eur"] = rentals_ams["rent"].apply(parse_eur)
rentals_ams["deposit_eur"] = rentals_ams["deposit"].apply(parse_eur)
rentals_ams["additional_costs_eur"] = rentals_ams["additionalCostsRaw"].apply(parse_eur)
rentals_ams["area_sqm"] = rentals_ams["areaSqm"].apply(parse_sqm)

num_cols_rentals = ["rent_eur", "deposit_eur", "additional_costs_eur", "area_sqm"]
rentals_ams[num_cols_rentals].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).round(
    2
)

In [ ]:
# Missingness on parsed + original key fields
cols = num_cols_rentals + [
    "postalCode",
    "latitude",
    "longitude",
    "propertyType",
    "furnish",
    "energyLabel",
]
miss = pd.DataFrame(
    {
        "n_missing": rentals_ams[cols].isna().sum(),
        "pct_missing": (rentals_ams[cols].isna().mean() * 100).round(2),
    }
)
miss

In [ ]:
# Outlier report — same IQR / z-score framework as Airbnb
rentals_outliers = pd.DataFrame({c: outlier_report(rentals_ams[c]) for c in num_cols_rentals}).T
rentals_outliers

In [ ]:
# Histograms — raw and log1p side-by-side
fig, axes = plt.subplots(len(num_cols_rentals), 2, figsize=(12, 3 * len(num_cols_rentals)))
for i, col in enumerate(num_cols_rentals):
    s = rentals_ams[col].dropna()
    axes[i, 0].hist(s, bins=60, color="darkorange")
    axes[i, 0].set_title(f"{col} — raw")
    axes[i, 1].hist(np.log1p(s), bins=60, color="darkorange")
    axes[i, 1].set_title(f"{col} — log1p")
plt.tight_layout()
plt.show()

In [ ]:
# Boxplots for each numeric rental field
fig, axes = plt.subplots(1, len(num_cols_rentals), figsize=(4 * len(num_cols_rentals), 4))
for ax, col in zip(axes, num_cols_rentals):
    ax.boxplot(rentals_ams[col].dropna(), vert=True)
    ax.set_title(col)
plt.tight_layout()
plt.show()

In [ ]:
# Top-10 rent outliers — sanity check on the high end
rentals_ams.nlargest(10, "rent_eur")[
    ["postalCode", "propertyType", "area_sqm", "rent_eur", "deposit_eur"]
]

### 2a. Why rent & area look bimodal — faceting by `propertyType`

Kamernet mixes two structurally different products: **rooms in shared
houses** (small, cheap) and **studios / apartments** (larger, more
expensive). Overlaying histograms per `propertyType` should resolve the
two peaks in `rent_eur` and `area_sqm` into clean per-category
distributions — confirming the latent variable rather than a data issue.

In [ ]:
# Property-type mix in the Amsterdam subset
rentals_ams["propertyType"].value_counts(dropna=False)

In [ ]:
# Overlayed histograms per propertyType for rent_eur and area_sqm.
# Clip at p99 so one tail doesn't flatten the others.
top_types = rentals_ams["propertyType"].value_counts().head(4).index.tolist()
palette = ["steelblue", "darkorange", "seagreen", "crimson"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for col, ax in zip(["rent_eur", "area_sqm"], axes):
    clip_hi = rentals_ams[col].quantile(0.99)
    bins = np.linspace(0, clip_hi, 60)
    for ptype, color in zip(top_types, palette):
        s = rentals_ams.loc[rentals_ams["propertyType"] == ptype, col].dropna().clip(upper=clip_hi)
        ax.hist(s, bins=bins, alpha=0.5, color=color, label=f"{ptype} (n={len(s)})")
    ax.set_title(f"{col} by propertyType (clipped at p99)")
    ax.set_xlabel(col)
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Bottom-10 rent outliers — look for suspiciously-low values (0 EUR, placeholder rents)
rentals_ams.nsmallest(10, "rent_eur")[["postalCode", "propertyType", "area_sqm", "rent_eur"]]

In [ ]:
# Rent-per-sqm — a cleaner outlier signal than raw rent, since it controls for size
rentals_ams["rent_per_sqm"] = rentals_ams["rent_eur"] / rentals_ams["area_sqm"].replace(0, np.nan)
print(outlier_report(rentals_ams["rent_per_sqm"]))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(rentals_ams["rent_per_sqm"].dropna(), bins=80, color="seagreen")
axes[0].set_title("Rent per sqm — raw")
axes[0].set_xlabel("EUR / m\u00b2")
axes[1].boxplot(rentals_ams["rent_per_sqm"].dropna(), vert=True)
axes[1].set_title("Rent per sqm — boxplot")
plt.tight_layout()
plt.show()

## 3. Cross-dataset comparison

Compare the Airbnb *monthly-equivalent* revenue (nightly × 365 × 0.66 / 12)
to the Kamernet monthly rent on the **same axis**.  This is a preview of the
Gold-layer answer — useful here to check that both distributions overlap
sensibly and that no outlier regime is dominating the comparison.

In [ ]:
OCCUPANCY_RATE = 0.66  # Amsterdam average (AirDNA)
airbnb_monthly = airbnb["price"].dropna() * 365 * OCCUPANCY_RATE / 12
rental_monthly = rentals_ams["rent_eur"].dropna()

summary = pd.DataFrame(
    {
        "airbnb_monthly_equiv": airbnb_monthly.describe(percentiles=[0.05, 0.5, 0.95]),
        "rental_monthly": rental_monthly.describe(percentiles=[0.05, 0.5, 0.95]),
    }
).round(0)
summary

In [ ]:
# Overlayed histograms, clipped at the 99th percentile for readability
clip = max(airbnb_monthly.quantile(0.99), rental_monthly.quantile(0.99))
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(
    airbnb_monthly.clip(upper=clip),
    bins=60,
    alpha=0.55,
    color="steelblue",
    label="Airbnb (monthly equiv.)",
)
ax.hist(
    rental_monthly.clip(upper=clip),
    bins=60,
    alpha=0.55,
    color="darkorange",
    label="Rental (monthly)",
)
ax.set_xlabel("EUR / month")
ax.set_ylabel("listings")
ax.set_title("Monthly revenue distribution — Airbnb vs Rental (clipped at p99)")
ax.legend()
plt.show()

### 3a. Before vs after IQR cap — the Silver effect

The raw Airbnb histogram above has a tail reaching tens of thousands of
EUR/month — e.g. a €1,264/night listing projects to ~€25k/month equivalent
at 66% occupancy, which obviously isn't real income. The ×365×0.66/12
transform amplifies nightly outliers ~20x, which is why the cap has to
happen on `nightly_price` *before* the monthly conversion.

This cell applies the same IQR cap Silver uses and overlays raw vs capped
on the Airbnb side, with Kamernet rent for reference. The capped version
is what the Gold layer's postcode averages actually use.

In [ ]:
# Recreate the Silver IQR cap inline, so this cell is self-contained
price = airbnb["price"].dropna()
q1, q3 = price.quantile([0.25, 0.75])
upper_fence = q3 + 1.5 * (q3 - q1)
print(f"IQR upper fence: EUR {upper_fence:,.0f}/night  (caps {(price > upper_fence).sum()} rows)")

airbnb_monthly_raw = price * 365 * OCCUPANCY_RATE / 12
airbnb_monthly_capped = price.clip(upper=upper_fence) * 365 * OCCUPANCY_RATE / 12

print(f"Raw    Airbnb monthly max: EUR {airbnb_monthly_raw.max():>10,.0f}")
print(f"Capped Airbnb monthly max: EUR {airbnb_monthly_capped.max():>10,.0f}")
print(f"Rental       monthly max: EUR {rental_monthly.max():>10,.0f}")

cap_monthly = upper_fence * 365 * OCCUPANCY_RATE / 12
n_capped = (price > upper_fence).sum()
print(f"Share capped: {n_capped / len(price) * 100:.1f}% of listings")

# Bulk view, linear axis up to just past the cap: capped as fill, raw as an
# outline drawn ON TOP (below the fence the two are identical, so the fill
# covers the outline — the cap only touches the tail, which piles up at the
# dashed cap line next to the raw outline's tail end)
fig, ax = plt.subplots(figsize=(11, 5))
bins_bulk = np.linspace(0, cap_monthly * 1.15, 80)
ax.hist(
    airbnb_monthly_capped, bins=bins_bulk, alpha=0.55, color="navy", label="Airbnb after IQR cap"
)
ax.hist(
    airbnb_monthly_raw,
    bins=bins_bulk,
    histtype="step",
    lw=2.5,
    color="steelblue",
    label="Airbnb raw (outline)",
)
ax.hist(rental_monthly, bins=bins_bulk, alpha=0.45, color="darkorange", label="Rental monthly")
ax.axvline(cap_monthly, color="navy", ls="--", lw=1, label="IQR cap -> monthly")
ax.set_title("Cross-dataset — raw vs IQR-capped Airbnb vs Rental")
ax.set_xlabel("EUR / month")
ax.set_ylabel("listings")
ax.legend()
plt.tight_layout()
plt.show()

## 4. Takeaways

- **Airbnb `price`** has a long right tail; IQR flags a few % as outliers
  → Silver caps at the IQR upper fence rather than dropping, to keep
    the postcode-level averages honest without being skewed by a handful
    of luxury listings.
- **Airbnb `zipcode`** missingness is non-trivial
  → Silver backfills via point-in-polygon against `post_codes.geojson`.
- **Kamernet `rent_eur`** also right-skewed; `rent_per_sqm` is a tighter
  outlier signal than raw rent because it controls for property size.
- **Kamernet currency / area fields** arrive as strings → parsed in Silver.
- **Geographic bbox** confirms both datasets are centred on Amsterdam —
  stray points outside the bbox are rare and can be left to the Silver
  filter (`city == 'Amsterdam'` for rentals; postcode-join for Airbnb).